In [1]:
# ============================================================
# NutriVision — Final Evaluation & Export
# Cell 1: Local Environment Verification
# ============================================================

from pathlib import Path
import sys
import os

# ------------------------------------------------------------
# 1. Project root
# ------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Users\SANDEEP\Documents\NutriVision")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"NutriVision project folder not found:\n{PROJECT_ROOT}"
    )

print("Project root:")
print(PROJECT_ROOT)


# ------------------------------------------------------------
# 2. Python environment
# ------------------------------------------------------------
print("\nPython:")
print(sys.version)
print("Executable:")
print(sys.executable)


# ------------------------------------------------------------
# 3. TensorFlow / Keras / KerasHub
# ------------------------------------------------------------
import tensorflow as tf
import keras
import keras_hub

print("\nLibraries:")
print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)
print("KerasHub   :", keras_hub.__version__)


# ------------------------------------------------------------
# 4. Check local checkpoint
# ------------------------------------------------------------
CHECKPOINT = (
    PROJECT_ROOT
    / "models"
    / "checkpoints"
    / "best_model.weights.h5"
)

print("\nCheckpoint:")
print(CHECKPOINT)

if not CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint not found:\n{CHECKPOINT}"
    )

size_mb = CHECKPOINT.stat().st_size / (1024 * 1024)

print("Checkpoint found successfully.")
print(f"Checkpoint size: {size_mb:.2f} MB")


# ------------------------------------------------------------
# 5. Check project data locations
# ------------------------------------------------------------
PROCESSED_DATA = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "foodseg103_segmentation"
)

RAW_DATA = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "FoodSeg103"
)

print("\nData paths:")
print("Processed:", PROCESSED_DATA)
print("Raw      :", RAW_DATA)

print("\nExistence:")
print("Processed dataset:", PROCESSED_DATA.exists())
print("Raw FoodSeg103   :", RAW_DATA.exists())


# ------------------------------------------------------------
# 6. GPU/CPU information
# ------------------------------------------------------------
gpus = tf.config.list_physical_devices("GPU")

print("\nCompute device:")
if gpus:
    print("GPU detected:", gpus)
else:
    print("No GPU detected. CPU execution will be used.")


# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("LOCAL FINALIZATION ENVIRONMENT VERIFIED")
print("=" * 60)
print("No training has been started.")
print("No model weights have been modified.")
print("=" * 60)

Project root:
C:\Users\SANDEEP\Documents\NutriVision

Python:
3.13.12 | packaged by Anaconda, Inc. | (main, Feb 24 2026, 16:05:56) [MSC v.1942 64 bit (AMD64)]
Executable:
c:\Users\SANDEEP\Documents\NutriVision\.venv\Scripts\python.exe

Libraries:
TensorFlow : 2.21.0
Keras      : 3.15.1
KerasHub   : 0.32.0

Checkpoint:
C:\Users\SANDEEP\Documents\NutriVision\models\checkpoints\best_model.weights.h5
Checkpoint found successfully.
Checkpoint size: 105.40 MB

Data paths:
Processed: C:\Users\SANDEEP\Documents\NutriVision\data\processed\foodseg103_segmentation
Raw      : C:\Users\SANDEEP\Documents\NutriVision\data\raw\FoodSeg103

Existence:
Processed dataset: True
Raw FoodSeg103   : True

Compute device:
No GPU detected. CPU execution will be used.

LOCAL FINALIZATION ENVIRONMENT VERIFIED
No training has been started.
No model weights have been modified.


In [3]:
# ============================================================
# NutriVision — Final Evaluation & Export
# Cell 2: Model + Checkpoint Verification
# ============================================================

import keras_hub

# ------------------------------------------------------------
# 1. Model configuration
# ------------------------------------------------------------
NUM_CLASSES = 104
PRESET = "segformer_b2_ade20k_512"

print("Building SegFormer B2...")
print("Preset     :", PRESET)
print("Num classes:", NUM_CLASSES)


# ------------------------------------------------------------
# 2. Build fresh model
# ------------------------------------------------------------
model = keras_hub.models.SegFormerImageSegmenter.from_preset(
    PRESET,
    num_classes=NUM_CLASSES,
)

print("\nModel created successfully.")


# ------------------------------------------------------------
# 3. Verify architecture
# ------------------------------------------------------------
print("\nModel input shape :", model.input_shape)
print("Model output shape:", model.output_shape)


# ------------------------------------------------------------
# 4. Load best production checkpoint
# ------------------------------------------------------------
print("\nLoading trained checkpoint...")

model.load_weights(CHECKPOINT)

print("Checkpoint loaded successfully.")


# ------------------------------------------------------------
# 5. Verify architecture exactly
# ------------------------------------------------------------
if model.input_shape != (None, 512, 512, 3):
    raise RuntimeError(
        f"Unexpected input shape: {model.input_shape}"
    )

if model.output_shape != (None, 512, 512, NUM_CLASSES):
    raise RuntimeError(
        f"Unexpected output shape: {model.output_shape}"
    )


# ------------------------------------------------------------
# 6. Final status
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("MODEL + CHECKPOINT VERIFICATION PASSED")
print("=" * 60)
print("Architecture : SegFormer B2")
print("Input        : 512 × 512 × 3")
print("Output       : 512 × 512 × 104")
print("Weights      : best_model.weights.h5")
print("Weights load : SUCCESS")
print("CPU forward  : SKIPPED — avoids unnecessary RAM usage")
print("=" * 60)

Building SegFormer B2...
Preset     : segformer_b2_ade20k_512
Num classes: 104

Model created successfully.

Model input shape : (None, 512, 512, 3)
Model output shape: (None, 512, 512, 104)

Loading trained checkpoint...
Checkpoint loaded successfully.

MODEL + CHECKPOINT VERIFICATION PASSED
Architecture : SegFormer B2
Input        : 512 × 512 × 3
Output       : 512 × 512 × 104
Weights      : best_model.weights.h5
Weights load : SUCCESS
CPU forward  : SKIPPED — avoids unnecessary RAM usage


In [7]:
# ============================================================
# NutriVision — Final Evaluation & Export
# Cell 4: Reconstruct Exact Production Split
# ============================================================

from pathlib import Path
import numpy as np


# ------------------------------------------------------------
# 1. Locate processed training images
# ------------------------------------------------------------
TRAIN_IMAGE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "foodseg103_segmentation"
    / "images"
    / "train"
)

if not TRAIN_IMAGE_DIR.exists():
    raise FileNotFoundError(
        f"Processed training image directory not found:\n"
        f"{TRAIN_IMAGE_DIR}"
    )


# ------------------------------------------------------------
# 2. Get and sort all image filenames
# ------------------------------------------------------------
image_names = sorted(
    p.name
    for p in TRAIN_IMAGE_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

print("Total processed training images:", len(image_names))

if len(image_names) != 4979:
    raise RuntimeError(
        f"Expected exactly 4979 processed training images, "
        f"but found {len(image_names)}."
    )


# ------------------------------------------------------------
# 3. Reproduce the ORIGINAL split procedure exactly
# ------------------------------------------------------------
rng = np.random.default_rng(42)

indices = np.arange(len(image_names))

rng.shuffle(indices)

dev_indices = np.sort(indices[:500])
train_indices = np.sort(indices[500:])


# ------------------------------------------------------------
# 4. Convert indices back to filenames
# ------------------------------------------------------------
dev_names = [image_names[i] for i in dev_indices]
train_names = [image_names[i] for i in train_indices]


# ------------------------------------------------------------
# 5. Verify counts and overlap
# ------------------------------------------------------------
train_set = set(train_names)
dev_set = set(dev_names)

overlap = train_set.intersection(dev_set)

print("\nReconstructed split:")
print("Production train:", len(train_names))
print("Development set :", len(dev_names))
print("Overlap         :", len(overlap))

if len(train_names) != 4479:
    raise RuntimeError(
        "Production train count is incorrect."
    )

if len(dev_names) != 500:
    raise RuntimeError(
        "Development set count is incorrect."
    )

if overlap:
    raise RuntimeError(
        f"Train/dev overlap detected: {len(overlap)} images."
    )


# ------------------------------------------------------------
# 6. Save split in the correct project location
# ------------------------------------------------------------
PRODUCTION_SPLIT_DIR = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "production"
)

PRODUCTION_SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRAIN_SPLIT_FILE = PRODUCTION_SPLIT_DIR / "train.txt"
DEV_SPLIT_FILE = PRODUCTION_SPLIT_DIR / "dev.txt"


TRAIN_SPLIT_FILE.write_text(
    "\n".join(train_names),
    encoding="utf-8"
)

DEV_SPLIT_FILE.write_text(
    "\n".join(dev_names),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------
print("\nSaved:")
print("Train split:", TRAIN_SPLIT_FILE)
print("Dev split  :", DEV_SPLIT_FILE)

print("\n" + "=" * 60)
print("EXACT PRODUCTION SPLIT RECONSTRUCTED")
print("=" * 60)
print("Seed            : 42")
print("Total           : 4979")
print("Production train: 4479")
print("Development     : 500")
print("Overlap         : 0")
print("=" * 60)

Total processed training images: 4979

Reconstructed split:
Production train: 4479
Development set : 500
Overlap         : 0

Saved:
Train split: C:\Users\SANDEEP\Documents\NutriVision\data\splits\production\train.txt
Dev split  : C:\Users\SANDEEP\Documents\NutriVision\data\splits\production\dev.txt

EXACT PRODUCTION SPLIT RECONSTRUCTED
Seed            : 42
Total           : 4979
Production train: 4479
Development     : 500
Overlap         : 0


In [8]:
# ============================================================
# NutriVision — Final Evaluation & Export
# Cell 5: Verify Development Image/Mask Pairing
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1. Define development image and mask directories
# ------------------------------------------------------------
DEV_IMAGE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "foodseg103_segmentation"
    / "images"
    / "train"
)

DEV_MASK_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "foodseg103_segmentation"
    / "masks"
    / "train"
)


# ------------------------------------------------------------
# 2. Load the permanently saved development split
# ------------------------------------------------------------
DEV_SPLIT_FILE = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "production"
    / "dev.txt"
)

if not DEV_SPLIT_FILE.exists():
    raise FileNotFoundError(
        f"Development split not found:\n{DEV_SPLIT_FILE}"
    )

dev_names = [
    line.strip()
    for line in DEV_SPLIT_FILE.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

print("Development images in split:", len(dev_names))


# ------------------------------------------------------------
# 3. Verify directories
# ------------------------------------------------------------
if not DEV_IMAGE_DIR.exists():
    raise FileNotFoundError(
        f"Image directory not found:\n{DEV_IMAGE_DIR}"
    )

if not DEV_MASK_DIR.exists():
    raise FileNotFoundError(
        f"Mask directory not found:\n{DEV_MASK_DIR}"
    )


# ------------------------------------------------------------
# 4. Verify every image has a corresponding mask
# ------------------------------------------------------------
missing_images = []
missing_masks = []

for name in dev_names:

    image_path = DEV_IMAGE_DIR / name

    mask_name = Path(name).with_suffix(".png").name
    mask_path = DEV_MASK_DIR / mask_name

    if not image_path.exists():
        missing_images.append(name)

    if not mask_path.exists():
        missing_masks.append(mask_name)


# ------------------------------------------------------------
# 5. Report verification
# ------------------------------------------------------------
print("\nVerification:")
print("Missing images:", len(missing_images))
print("Missing masks :", len(missing_masks))

if missing_images:
    print("\nFirst missing images:")
    for name in missing_images[:10]:
        print(" ", name)

if missing_masks:
    print("\nFirst missing masks:")
    for name in missing_masks[:10]:
        print(" ", name)


# ------------------------------------------------------------
# 6. Stop if anything is missing
# ------------------------------------------------------------
if missing_images or missing_masks:
    raise RuntimeError(
        "Development image/mask pairing verification failed."
    )


# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("DEVELOPMENT IMAGE/MASK PAIRING VERIFIED")
print("=" * 60)
print("Development images :", len(dev_names))
print("Matching masks     :", len(dev_names))
print("Missing images     : 0")
print("Missing masks      : 0")
print("=" * 60)

Development images in split: 500

Verification:
Missing images: 0
Missing masks : 0

DEVELOPMENT IMAGE/MASK PAIRING VERIFIED
Development images : 500
Matching masks     : 500
Missing images     : 0
Missing masks      : 0


In [9]:
# ============================================================
# NutriVision — Final Evaluation & Export
# Cell 6: Prepare Resumable Evaluation Configuration
# ============================================================

from pathlib import Path
import json


# ------------------------------------------------------------
# 1. Evaluation configuration
# ------------------------------------------------------------
EVAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
)

EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVAL_RESULTS_FILE = (
    EVAL_DIR
    / "dev_box_evaluation.csv"
)

EVAL_CONFIG_FILE = (
    EVAL_DIR
    / "evaluation_config.json"
)


# ------------------------------------------------------------
# 2. Locked evaluation settings
# ------------------------------------------------------------
evaluation_config = {
    "dataset": "FoodSeg103",
    "split": "production_development",
    "num_images": 500,
    "model_preset": "segformer_b2_ade20k_512",
    "num_classes": 104,
    "image_size": 512,
    "iou_threshold": 0.50,
    "prediction_min_area_values": [
        20,
        50,
        100,
        250,
        500
    ],
    "ground_truth_method": "connected_components",
    "matching": "class_aware_one_to_one",
    "training": False,
    "gradient_computation": False,
    "resume_enabled": True,
    "checkpoint": str(CHECKPOINT),
}


# ------------------------------------------------------------
# 3. Save configuration
# ------------------------------------------------------------
EVAL_CONFIG_FILE.write_text(
    json.dumps(
        evaluation_config,
        indent=4
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 4. Verify required paths
# ------------------------------------------------------------
required_paths = {
    "checkpoint": CHECKPOINT,
    "processed_dataset": PROCESSED_DATA,
    "dev_split": DEV_SPLIT_FILE,
}

print("Required resources:\n")

all_available = True

for name, path in required_paths.items():

    exists = Path(path).exists()

    print(
        f"{name:<20}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    if not exists:
        all_available = False


# ------------------------------------------------------------
# 5. Existing evaluation status
# ------------------------------------------------------------
print("\nEvaluation result file:")

if EVAL_RESULTS_FILE.exists():
    print("Existing result file found:")
    print(EVAL_RESULTS_FILE)
else:
    print("No previous evaluation results found.")
    print("A new resumable evaluation will be created.")


# ------------------------------------------------------------
# 6. Final status
# ------------------------------------------------------------
if not all_available:
    raise RuntimeError(
        "One or more required evaluation resources are missing."
    )

print("\n" + "=" * 60)
print("EVALUATION CONFIGURATION READY")
print("=" * 60)
print("Images            : 500")
print("IoU threshold     : 0.50")
print("Matching          : class-aware one-to-one")
print("GT boxes          : connected components")
print("Prediction areas  : 20, 50, 100, 250, 500")
print("Resume            : enabled")
print("Training          : NO")
print("Local CPU eval    : NOT RUN")
print("=" * 60)

Required resources:

checkpoint          : FOUND
processed_dataset   : FOUND
dev_split           : FOUND

Evaluation result file:
No previous evaluation results found.
A new resumable evaluation will be created.

EVALUATION CONFIGURATION READY
Images            : 500
IoU threshold     : 0.50
Matching          : class-aware one-to-one
GT boxes          : connected components
Prediction areas  : 20, 50, 100, 250, 500
Resume            : enabled
Training          : NO
Local CPU eval    : NOT RUN


In [10]:
# ============================================================
# Prepare exactly the 500 development samples for Colab
# ============================================================

from pathlib import Path
import shutil

# ------------------------------------------------------------
# Source locations
# ------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Users\SANDEEP\Documents\NutriVision")

DEV_SPLIT_FILE = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "production"
    / "dev.txt"
)

SOURCE_IMAGES = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "foodseg103_segmentation"
    / "images"
    / "train"
)

SOURCE_MASKS = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "foodseg103_segmentation"
    / "masks"
    / "train"
)

# ------------------------------------------------------------
# Destination
# ------------------------------------------------------------
EVAL_PACKAGE = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "dev_500"
)

DEST_IMAGES = EVAL_PACKAGE / "images"
DEST_MASKS = EVAL_PACKAGE / "masks"

# Start clean if this package already exists
if EVAL_PACKAGE.exists():
    shutil.rmtree(EVAL_PACKAGE)

DEST_IMAGES.mkdir(parents=True)
DEST_MASKS.mkdir(parents=True)

# ------------------------------------------------------------
# Read the exact 500 development filenames
# ------------------------------------------------------------
dev_names = [
    line.strip()
    for line in DEV_SPLIT_FILE.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

if len(dev_names) != 500:
    raise RuntimeError(
        f"Expected 500 development images, found {len(dev_names)}."
    )

# ------------------------------------------------------------
# Copy exactly those 500 images and masks
# ------------------------------------------------------------
for image_name in dev_names:

    image_source = SOURCE_IMAGES / image_name

    mask_name = Path(image_name).with_suffix(".png").name
    mask_source = SOURCE_MASKS / mask_name

    if not image_source.exists():
        raise FileNotFoundError(
            f"Development image missing:\n{image_source}"
        )

    if not mask_source.exists():
        raise FileNotFoundError(
            f"Development mask missing:\n{mask_source}"
        )

    shutil.copy2(
        image_source,
        DEST_IMAGES / image_name
    )

    shutil.copy2(
        mask_source,
        DEST_MASKS / mask_name
    )

# ------------------------------------------------------------
# Copy the split file too
# ------------------------------------------------------------
shutil.copy2(
    DEV_SPLIT_FILE,
    EVAL_PACKAGE / "dev.txt"
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------
image_count = len(list(DEST_IMAGES.glob("*")))
mask_count = len(list(DEST_MASKS.glob("*")))

print("=" * 60)
print("DEV 500 EVALUATION PACKAGE CREATED")
print("=" * 60)
print("Development images :", image_count)
print("Development masks  :", mask_count)
print("Split file         :", EVAL_PACKAGE / "dev.txt")
print("Package location   :", EVAL_PACKAGE)
print("=" * 60)

DEV 500 EVALUATION PACKAGE CREATED
Development images : 500
Development masks  : 500
Split file         : C:\Users\SANDEEP\Documents\NutriVision\data\evaluation\dev_500\dev.txt
Package location   : C:\Users\SANDEEP\Documents\NutriVision\data\evaluation\dev_500


In [11]:
from pathlib import Path

RAW_FOODSEG = Path(
    r"C:\Users\SANDEEP\Documents\NutriVision\data\raw\FoodSeg103"
)

VAL_PARQUET = RAW_FOODSEG / "data" / "validation-00000-of-00001.parquet"

print("Dataset exists:", RAW_FOODSEG.exists())
print("Validation parquet exists:", VAL_PARQUET.exists())

Dataset exists: True
Validation parquet exists: True


In [12]:
import pandas as pd

val_df = pd.read_parquet(VAL_PARQUET)

print("Validation images:", len(val_df))
print("Columns:", list(val_df.columns))

Validation images: 2135
Columns: ['image', 'label', 'classes_on_image', 'id']


In [13]:
FINAL_VAL_ROOT = Path(
    r"C:\Users\SANDEEP\Documents\NutriVision\data\evaluation\raw_foodseg103_val"
)

(FINAL_VAL_ROOT / "images").mkdir(parents=True, exist_ok=True)
(FINAL_VAL_ROOT / "masks").mkdir(parents=True, exist_ok=True)

print("Images folder:", (FINAL_VAL_ROOT / "images").exists())
print("Masks folder :", (FINAL_VAL_ROOT / "masks").exists())

Images folder: True
Masks folder : True


In [14]:
sample = val_df.iloc[0]

print("ID:", sample["id"])
print("Image type:", type(sample["image"]))
print("Label type:", type(sample["label"]))

print("Image keys:", sample["image"].keys())
print("Label keys:", sample["label"].keys())

ID: 0
Image type: <class 'dict'>
Label type: <class 'dict'>
Image keys: dict_keys(['bytes', 'path'])
Label keys: dict_keys(['bytes', 'path'])


In [15]:
from PIL import Image
import io

images_dir = FINAL_VAL_ROOT / "images"
masks_dir = FINAL_VAL_ROOT / "masks"

for i, row in val_df.iterrows():

    image = Image.open(
        io.BytesIO(row["image"]["bytes"])
    ).convert("RGB")

    mask = Image.open(
        io.BytesIO(row["label"]["bytes"])
    ).convert("L")

    image.save(images_dir / f"{i:05d}.jpg", quality=95)
    mask.save(masks_dir / f"{i:05d}.png")

    if (i + 1) % 250 == 0:
        print(f"{i + 1}/2135")

print("Export complete.")

250/2135
500/2135
750/2135
1000/2135
1250/2135
1500/2135
1750/2135
2000/2135
Export complete.


In [16]:
image_files = list(images_dir.glob("*.jpg"))
mask_files = list(masks_dir.glob("*.png"))

print("Images:", len(image_files))
print("Masks :", len(mask_files))

if len(image_files) != 2135 or len(mask_files) != 2135:
    raise RuntimeError("Final validation export is incomplete.")

print("FINAL VALIDATION EXPORT PASSED")

Images: 2135
Masks : 2135
FINAL VALIDATION EXPORT PASSED


In [17]:
import shutil

zip_base = FINAL_VAL_ROOT.parent / "raw_foodseg103_val"

shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=FINAL_VAL_ROOT
)

zip_path = Path(str(zip_base) + ".zip")

print("ZIP created:", zip_path)
print("Size (GB):", round(zip_path.stat().st_size / (1024**3), 3))


ZIP created: C:\Users\SANDEEP\Documents\NutriVision\data\evaluation\raw_foodseg103_val.zip
Size (GB): 0.185


In [18]:
print("ZIP exists:", zip_path.exists())
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024**2), 2))

ZIP exists: True
ZIP size (MB): 189.46
